# Credit Card Fraud Detection

End-to-end pipeline on the Kaggle Credit Card Fraud Detection dataset.

**Goal:** classify each transaction as genuine (0) or fraudulent (1).

**Challenge:** the positive class is only ~0.17 % of the data — plain accuracy is meaningless.

## 1. Setup

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, roc_curve,
)
from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
RANDOM_STATE = 42

## 2. Load the data

In [ ]:
df = pd.read_csv('../data/creditcard.csv')
print(df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.isnull().sum().sum()

## 3. EDA — class imbalance

In [ ]:
counts = df['Class'].value_counts()
print(counts)
print(f"Fraud rate: {counts[1] / len(df):.4%}")

sns.countplot(x='Class', data=df, palette=['#2ecc71', '#e74c3c'])
plt.title('Class Distribution (0 = Genuine, 1 = Fraud)')
plt.show()

Fraud is **extremely rare**. A model that always predicts 0 would score 99.83 % accuracy and catch zero frauds, so we will focus on **recall** and **ROC-AUC**.

## 4. Transaction amount analysis

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(df['Amount'], bins=80, color='#3498db')
plt.xlim(0, df['Amount'].quantile(0.99))
plt.title('Transaction Amount Distribution')
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
sns.boxplot(x='Class', y='Amount', data=df, palette=['#2ecc71', '#e74c3c'])
plt.ylim(0, df['Amount'].quantile(0.99))
plt.title('Amount by Class')
plt.show()

df.groupby('Class')['Amount'].describe()

## 5. Correlation

In [ ]:
plt.figure(figsize=(12, 9))
sns.heatmap(df.corr(), cmap='coolwarm', center=0)
plt.title('Feature Correlation')
plt.show()

V1–V28 are PCA components, so they are nearly uncorrelated with each other by construction. A few of them (V11, V4, V2 positively; V14, V12, V10 negatively) show the strongest correlation with `Class` — these are the most useful for fraud detection.

## 6. Feature prep + stratified split

In [ ]:
X = df.drop('Class', axis=1).copy()
y = df['Class'].copy()

scaler = StandardScaler()
X[['Time', 'Amount']] = scaler.fit_transform(X[['Time', 'Amount']])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y,
)
print('train:', X_train.shape, 'test:', X_test.shape)
print('train class counts:', np.bincount(y_train))

## 7. Handle imbalance with SMOTE

SMOTE synthesises new minority-class samples in feature space. Apply it **only to the training set** — the test set must keep the real production distribution.

In [ ]:
smote = SMOTE(random_state=RANDOM_STATE)
X_res, y_res = smote.fit_resample(X_train, y_train)
print('after SMOTE:', np.bincount(y_res))

## 8. Train models

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, n_jobs=-1)
lr.fit(X_res, y_res)

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_res, y_res)

## 9. Evaluate

In [ ]:
def evaluate(name, model):
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, prob)
    print(f'\n===== {name} =====')
    print(classification_report(y_test, pred, digits=4))
    print(f'ROC-AUC: {auc:.4f}')

    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Genuine','Fraud'], yticklabels=['Genuine','Fraud'])
    plt.title(f'Confusion Matrix — {name}')
    plt.xlabel('Predicted'); plt.ylabel('Actual')
    plt.show()
    return prob, auc

lr_prob, lr_auc = evaluate('Logistic Regression', lr)
rf_prob, rf_auc = evaluate('Random Forest', rf)

In [ ]:
for name, prob, auc in [('LR', lr_prob, lr_auc), ('RF', rf_prob, rf_auc)]:
    fpr, tpr, _ = roc_curve(y_test, prob)
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})')
plt.plot([0,1], [0,1], 'k--', alpha=0.4)
plt.xlabel('FPR'); plt.ylabel('TPR'); plt.title('ROC Curves')
plt.legend(loc='lower right')
plt.show()

## 10. Business takeaways

- Fraud is ~0.17 % of transactions — **accuracy is the wrong metric**.
- **Recall** is the priority: every missed fraud is a direct loss.
- SMOTE lets the model learn the minority class without throwing away genuine data.
- Random Forest beats the linear baseline on ROC-AUC and is the better production candidate.
- Next steps: tune the decision threshold for the precision/recall trade-off your business actually wants, try gradient boosting (XGBoost / LightGBM), and monitor drift over time.